In [278]:
import os
from pathlib import Path

# Change directory
# Modify this cell to insure that the output shows the correct path.
# Define all paths relative to the project root shown in the cell output
project_root = "/Users/conradlz/Documents/webclones/freqtrade"
i = 0
try:
    os.chdir(project_root)
    if not Path("LICENSE").is_file():
        i = 0
        while i < 4 and (not Path("LICENSE").is_file()):
            os.chdir(Path(Path.cwd(), "../"))
            i += 1
        project_root = Path.cwd()
except FileNotFoundError:
    print("Please define the project root relative to the current directory")
print(Path.cwd())


/Users/conradlz/Documents/webclones/freqtrade


In [279]:
from freqtrade.configuration import Configuration
from freqtrade.enums import RunMode
from freqtrade.resolvers import ExchangeResolver

# Initialize configuration
config = Configuration.from_files(["user_data/config.json"])

# Define constants for WarriorMomentum strategy
config["timeframe"] = "5m"
config["strategy"] = "WarriorMomentum"
config["stake_currency"] = "USD"
config["dry_run"] = True
config["dataformat_ohlcv"] = "feather"
config["runmode"] = RunMode.BACKTEST

# Data location
data_location = config["datadir"]

print(f"Strategy: {config['strategy']}")
print(f"Timeframe: {config['timeframe']}")
print(f"Data directory: {data_location}")
print(f"Stake currency: {config['stake_currency']}")
print(f"Max open trades: {config['max_open_trades']}")
print(f"Stake amount: {config['stake_amount']}")


2025-07-18 17:01:40,882 - freqtrade.configuration.load_config - INFO - Using config: user_data/config.json ...

2025-07-18 17:01:40,889 - freqtrade.loggers - INFO - Enabling colorized output.

2025-07-18 17:01:40,890 - freqtrade.loggers - INFO - Logfile configured

2025-07-18 17:01:40,891 - freqtrade.loggers - INFO - Verbosity set to 0

2025-07-18 17:01:40,893 - freqtrade.configuration.configuration - INFO - Using user-data directory: /Users/conradlz/Documents/webclones/freqtrade/user_data ...

2025-07-18 17:01:40,894 - freqtrade.configuration.configuration - INFO - Using data directory: /Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken ...

2025-07-18 17:01:40,895 - freqtrade.exchange.check_exchange - INFO - Checking exchange...

2025-07-18 17:01:40,903 - freqtrade.exchange.check_exchange - INFO - Exchange "kraken" is officially supported by the Freqtrade development team.

Strategy: WarriorMomentum
Timeframe: 5m
Data directory: /Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken
Stake currency: USD
Max open trades: 5
Stake amount: unlimited


In [ ]:
import glob
import re
from pathlib import Path

def find_usdt_pairs(data_dir, timeframe="5m"):
    """Find all USDT pairs available in the data directory."""
    data_path = Path(data_dir)
    usdt_pairs = []

    # Search for feather files with USDT pairs
    pattern = f"*USD-{timeframe}.feather"

    # Search in all subdirectories (different exchanges)
    print(data_path)
    if data_path.is_dir():
        files = list(data_path.glob(pattern))
        print(files)
        for file in files:
            # Extract pair name from filename
            pair_name = file.stem.replace(f"-{timeframe}", "").replace("_", "/")
            print(pair_name)
            if pair_name not in usdt_pairs:
                usdt_pairs.append(pair_name)
    else:
        print(f"Not a directory: {data_path}")

    return sorted(usdt_pairs)

# Find all USDT pairs
usdt_pairs = find_usdt_pairs(data_location, config["timeframe"])

print(f"Found {len(usdt_pairs)} USD pairs:")
for i, pair in enumerate(usdt_pairs, 1):
    print(f"{i:2d}. {pair}")

if not usdt_pairs:
    print("No USD pairs found. Please download data first using:")
    print("freqtrade download-data --exchange binance --pairs BTC/USDT ETH/USDT --timeframes 5m 1d")
    print("Or use the exchange's available pairs list instead.")


/Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken
[]
Found 0 USD pairs:
No USDT pairs found. Please download data first using:
freqtrade download-data --exchange binance --pairs BTC/USDT ETH/USDT --timeframes 5m 1d
Or use the exchange's available pairs list instead.


In [281]:
# If no local data found, get pairs from exchange
if not usdt_pairs:
    try:
        # Initialize exchange
        exchange = ExchangeResolver.load_exchange(config, validate=False)

        # Get all available pairs
        all_pairs = exchange.get_markets().keys()

        # Filter for USDT pairs
        usdt_pairs = [pair for pair in all_pairs if pair.endswith('/USDT')]
        usdt_pairs = sorted(usdt_pairs)

        print(f"Found {len(usdt_pairs)} USDT pairs from exchange:")
        for i, pair in enumerate(usdt_pairs[:20], 1):  # Show first 20
            print(f"{i:2d}. {pair}")

        if len(usdt_pairs) > 20:
            print(f"... and {len(usdt_pairs) - 20} more pairs")

    except Exception as e:
        print(f"Error getting pairs from exchange: {e}")
        # Fallback to common USDT pairs
        usdt_pairs = [
            "BTC/USDT", "ETH/USDT", "BNB/USDT", "ADA/USDT", "XRP/USDT",
            "SOL/USDT", "DOT/USDT", "DOGE/USDT", "AVAX/USDT", "SHIB/USDT",
            "MATIC/USDT", "LTC/USDT", "UNI/USDT", "LINK/USDT", "ATOM/USDT",
            "XLM/USDT", "BCH/USDT", "NEAR/USDT", "ALGO/USDT", "VET/USDT"
        ]
        print(f"Using fallback list of {len(usdt_pairs)} common USDT pairs")

print(f"\nTotal USDT pairs to analyze: {len(usdt_pairs)}")


2025-07-18 17:01:41,042 - freqtrade.exchange.exchange - INFO - Instance is running with dry_run enabled

2025-07-18 17:01:41,045 - freqtrade.exchange.exchange - INFO - Using CCXT 4.4.94

2025-07-18 17:01:41,079 - freqtrade.exchange.exchange - INFO - Using Exchange "Kraken"

2025-07-18 17:01:41,081 - freqtrade.resolvers.exchange_resolver - INFO - Using resolved exchange 'Kraken'...

2025-07-18 17:01:41,083 - freqtrade.exchange.exchange - INFO - Markets were not loaded. Loading them now..

Found 36 USDT pairs from exchange:
 1. ADA/USDT
 2. AI16Z/USDT
 3. ALGO/USDT
 4. APE/USDT
 5. ATOM/USDT
 6. AVAX/USDT
 7. BCH/USDT
 8. BERA/USDT
 9. BNB/USDT
10. BTC/USDT
11. CRO/USDT
12. DAI/USDT
13. DOGE/USDT
14. DOT/USDT
15. ETH/USDT
16. EURR/USDT
17. FARTCOIN/USDT
18. LINK/USDT
19. LTC/USDT
20. MANA/USDT
... and 16 more pairs

Total USDT pairs to analyze: 36


In [282]:
from freqtrade.enums.candletype import CandleType
from freqtrade.optimize.backtesting import Backtesting
from freqtrade.data.history import load_pair_history
import asyncio
import nest_asyncio

# Enable nested event loops for Jupyter notebooks
nest_asyncio.apply()

# Configure backtest settings for multi-pair analysis
config_backtest = config.copy()
config_backtest.update({
    "timerange": "",  # Use all available data
    "stake_currency": "USDT",
    "dry_run": True,
    "dataformat_ohlcv": "feather",
    "runmode": RunMode.BACKTEST,

    "pairlists": [{"method": "StaticPairList", "pairlist": usdt_pairs}],
    "position_adjustment_enable": True,  # Enable for WarriorMomentum strategy
    "max_open_trades": 10,  # Increase for multi-pair testing,
    "candle_type_def": CandleType.SPOT,
})

print("=== BACKTEST CONFIGURATION ===")
print(f"Strategy: {config_backtest['strategy']}")
print(f"Timeframe: {config_backtest['timeframe']}")
print(f"Pairs: {len(usdt_pairs)} USDT pairs")
print(f"Max open trades: {config_backtest['max_open_trades']}")
print(f"Stake amount: {config_backtest['stake_amount']} {config_backtest['stake_currency']}")
print(f"Position adjustment: {config_backtest['position_adjustment_enable']}")
print(f"Data directory: {config_backtest['datadir']}")
print(f"Data format: {config_backtest['dataformat_ohlcv']}")

# Initialize exchange if not already done
if 'exchange' not in locals():
    exchange = ExchangeResolver.load_exchange(config_backtest, validate=False)

print("\n=== INITIALIZING BACKTEST ===")
print("Setting up backtesting engine...")

# Initialize backtesting
backtesting = Backtesting(config_backtest, exchange)

print("Backtest engine initialized successfully!")


=== BACKTEST CONFIGURATION ===
Strategy: WarriorMomentum
Timeframe: 5m
Pairs: 36 USDT pairs
Max open trades: 10
Stake amount: unlimited USDT
Position adjustment: True
Data directory: /Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken
Data format: feather

=== INITIALIZING BACKTEST ===
Setting up backtesting engine...


2025-07-18 17:01:42,995 - freqtrade.resolvers.iresolver - INFO - Using resolved strategy WarriorMomentum from '/Users/conradlz/Documents/webclones/freqtrade/user_data/strategies/WarriorMomentum.py'...

2025-07-18 17:01:42,997 - freqtrade.strategy.hyper - INFO - Found no parameter file.

2025-07-18 17:01:42,998 - freqtrade.resolvers.strategy_resolver - INFO - Override strategy 'timeframe' with value in config file: 5m.

2025-07-18 17:01:42,999 - freqtrade.resolvers.strategy_resolver - INFO - Override strategy 'stake_currency' with value in config file: USDT.

...

2025-07-18 17:01:43,170 - freqtrade.plugins.pairlist.IPairList - WARNING - Pair IOTA/USDT is not compatible with exchange Kraken. Removing it from whitelist..

2025-07-18 17:01:43,171 - freqtrade.plugins.pairlist.IPairList - WARNING - Pair NEO/USDT is not compatible with exchange Kraken. Removing it from whitelist..

2025-07-18 17:01:43,172 - freqtrade.plugins.pairlist.IPairList - WARNING - Pair NXS/USDT is not compatible with exchange Kraken. Removing it from whitelist..

2025-07-18 17:01:43,175 - freqtrade.optimize.backtesting - INFO - Using fee 0.4000% - worst case fee from exchange (lowest tier).

Backtest engine initialized successfully!


In [283]:
print("Running backtest...")
print(f"Strategy: {config_backtest['strategy']}")
print(f"Timeframe: {config_backtest['timeframe']}")

# Run the full backtest process
backtesting.start()

print("Backtest completed!")
print(f"Results available in backtesting.results")


Running backtest...
Strategy: WarriorMomentum
Timeframe: 5m


2025-07-18 17:01:43,183 - freqtrade.data.history.history_utils - INFO - Using indicator startup period: 200 ...

2025-07-18 17:01:43,202 - freqtrade.data.converter.converter - INFO - Missing data fillup for ALGO/USDT, 5m: before: 3562 - after: 8900 - 149.86%

2025-07-18 17:01:43,214 - freqtrade.data.converter.converter - INFO - Missing data fillup for ATOM/USDT, 5m: before: 2157 - after: 8889 - 312.10%

2025-07-18 17:01:43,236 - freqtrade.data.converter.converter - INFO - Missing data fillup for BCH/USDT, 5m: before: 3049 - after: 8901 - 191.93%

...

                                                           MIXED TAG STATS                                                            
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Enter Tag ┃        Exit Reason ┃ Trades ┃ Avg Profit % ┃ Tot Profit USDT ┃ Tot Profit % ┃    Avg Duration ┃  Win  Draw  Loss  Win% ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│           │ trailing_stop_loss │     13 │         3.31 │        2169.348 │         4.34 │ 2 days, 3:11:00 │   13     0     0   100 │
│           │         force_exit │      1 │         1.25 │          64.036 │         0.13 │        12:45:00 │    1     0     0   100 │
│           │          stop_loss │      1 │       -10.71 │        -551.383 │         -1.1 │        23:15:00 │    0     0     1     0 │
│     TOTAL │                    │     15 │         2.24 │        1682.001 │         3.36 │ 1 day, 22:45:00 │   14     0     1  93.3 │
└───────────┴────────────────────┴────────┴──────────────┴─────────────────┴──────────────┴─────────────────┴────────────────────────┘

                          SUMMARY METRICS                          
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric                        ┃ Value                           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Backtesting from              │ 2025-06-17 13:40:00             │
│ Backtesting to                │ 2025-07-17 19:15:00             │
│ Trading Mode                  │ Spot                            │
│ Max open trades               │ 9                               │
│                               │                                 │
│ Total/Daily Avg Trades        │ 15 / 0.5                        │
│ Starting balance              │ 50000 USDT                      │
│ Final balance                 │ 0 USDT                          │
│ Absolute profit               │ 1682.001 USDT                   │
│ Total profit %                │ 3.36%                           │
│ CAGR %                        │ -100.00%                        │
│ Sortino                       │ -100.00                         │
│ Sharpe                        │ 4.83                            │
│ Calmar                        │ 201.54                          │
│ SQN                           │ 1.89                            │
│ Profit factor                 │ 4.05                            │
│ Expectancy (Ratio)            │ 112.13 (0.20)                   │
│ Avg. daily profit %           │ 0.11%                           │
│ Avg. stake amount             │ 5046.938 USDT                   │
│ Total trade volume            │ 154319.792 USDT                 │
│                               │                                 │
│ Best Pair                     │ XRP/USDT 1.56%                  │
│ Worst Pair                    │ ALGO/USDT -0.61%                │
│ Best trade                    │ LINK/USDT 10.78%                │
│ Worst trade                   │ ALGO/USDT -10.71%               │
│ Best day                      │ 1217.078 USDT                   │
│ Worst day                     │ -551.383 USDT                   │
│ Days win/draw/lose            │ 8 / 9 / 1                       │
│ Min/Max/Avg. Duration Winners │ 0d 00:45 / 10d 04:25 / 2d 00:26 │
│ Min/Max/Avg. Duration Losers  │ 0d 23:15 / 0d 23:15 / 0d 23:15  │
│ Max Consecutive Wins / Loss   │ 10 / 1                          │
│ Rejected Entry signals        │ 0                               │
│ Entry/Exit Timeouts           │ 0 / 0                           │
│                               │                                 │
│ Min balance                   │ 50123.18 USDT                   │
│ Max balance                   │ 51873.099 USDT                  │
│ Max % of account underwater   │ 1.06%                           │
│ Absolute Drawdown (Account)   │ 1.06%                           │
│ Absolute Drawdown             │ 551.383 USDT                    │
│ Drawdown high                 │ 1873.099 USDT                   │
│ Drawdown low                  │ 1321.716 USDT                   │
│ Drawdown Start                │ 2025-07-14 10:05:00             │
│ Drawdown End                  │ 2025-07-15 03:20:00             │
│ Market change                 │ 29.50%                          │
└───────────────────────────────┴─────────────────────────────────┘


Backtested 2025-06-17 13:40:00 -> 2025-07-17 19:15:00 | Max open trades : 9


                                                              STRATEGY SUMMARY                                                               
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃        Strategy ┃ Trades ┃ Avg Profit % ┃ Tot Profit USDT ┃ Tot Profit % ┃    Avg Duration ┃  Win  Draw  Loss  Win% ┃            Drawdown ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ WarriorMomentum │     15 │         2.24 │        1682.001 │         3.36 │ 1 day, 22:45:00 │   14     0     1  93.3 │ 551.383 USDT  1.06% │
└─────────────────┴────────┴──────────────┴─────────────────┴──────────────┴─────────────────┴────────────────────────┴─────────────────────┘

Backtest completed!
Results available in backtesting.results


In [289]:
import pandas as pd
import numpy as np

# Check if we have results to analyze
if 'results_df' in locals() and len(results_df) > 0:
    print("=== DETAILED PERFORMANCE METRICS ===")

    # Basic statistics
    total_trades = len(results_df)
    winning_trades = len(results_df[results_df['profit_ratio'] > 0])
    losing_trades = len(results_df[results_df['profit_ratio'] <= 0])
    win_rate = (winning_trades / total_trades) * 100

    # Profit statistics
    total_profit = results_df['profit_abs'].sum()
    avg_profit_per_trade = results_df['profit_ratio'].mean() * 100
    median_profit = results_df['profit_ratio'].median() * 100
    best_trade = results_df['profit_ratio'].max() * 100
    worst_trade = results_df['profit_ratio'].min() * 100

    # Calculate additional metrics
    results_df['cumulative_profit'] = results_df['profit_abs'].cumsum()
    peak = results_df['cumulative_profit'].cummax()
    drawdown = results_df['cumulative_profit'] - peak
    max_drawdown = drawdown.min()

    # Sharpe ratio (simplified)
    returns = results_df['profit_ratio']
    sharpe_ratio = returns.mean() / returns.std() if returns.std() > 0 else 0

    # Expectancy
    avg_win = results_df[results_df['profit_ratio'] > 0]['profit_ratio'].mean() if winning_trades > 0 else 0
    avg_loss = results_df[results_df['profit_ratio'] <= 0]['profit_ratio'].mean() if losing_trades > 0 else 0
    expectancy = (win_rate/100 * avg_win) + ((100-win_rate)/100 * avg_loss)

    print(f"📊 Total Trades: {total_trades}")
    print(f"🎯 Winning Trades: {winning_trades} ({win_rate:.2f}%)")
    print(f"❌ Losing Trades: {losing_trades} ({100-win_rate:.2f}%)")
    print(f"💰 Total Profit: {total_profit:.2f} {config_backtest['stake_currency']}")
    print(f"📈 Average Profit per Trade: {avg_profit_per_trade:.2f}%")
    print(f"📊 Median Profit per Trade: {median_profit:.2f}%")
    print(f"🚀 Best Trade: {best_trade:.2f}%")
    print(f"💥 Worst Trade: {worst_trade:.2f}%")
    print(f"📉 Maximum Drawdown: {max_drawdown:.2f} {config_backtest['stake_currency']}")
    print(f"📊 Sharpe Ratio: {sharpe_ratio:.3f}")
    print(f"🎯 Expectancy: {expectancy:.4f}")

    # Trade duration analysis
    if 'trade_duration' in results_df.columns:
        avg_duration = results_df['trade_duration'].mean()
        print(f"⏱️ Average Trade Duration: {avg_duration:.0f} minutes")

    print("\n=== PERFORMANCE BY PAIR ===")
    # Performance by pair
    pair_stats = results_df.groupby('pair').agg({
        'profit_ratio': ['count', 'mean', 'sum', 'std'],
        'profit_abs': 'sum',
        'trade_duration': 'mean' if 'trade_duration' in results_df.columns else lambda x: 0
    }).round(4)

    # Flatten column names
    pair_stats.columns = ['Trades', 'Avg_Profit_%', 'Total_Profit_%', 'Profit_Std', 'Total_Profit_Abs', 'Avg_Duration']
    pair_stats['Avg_Profit_%'] *= 100
    pair_stats['Total_Profit_%'] *= 100
    pair_stats['Win_Rate_%'] = results_df.groupby('pair')['profit_ratio'].apply(lambda x: (x > 0).mean() * 100)

    # Sort by total profit
    pair_stats = pair_stats.sort_values('Total_Profit_Abs', ascending=False)

    print("Top 10 performing pairs:")
    display_cols = ['Trades', 'Win_Rate_%', 'Avg_Profit_%', 'Total_Profit_Abs']
    print(pair_stats[display_cols].head(10).to_string())

    print("\nBottom 10 performing pairs:")
    print(pair_stats[display_cols].tail(10).to_string())

else:
    print("⚠️ No results available for analysis.")
    print("Please run the backtest first or check if data is available for the selected pairs.")


⚠️ No results available for analysis.
Please run the backtest first or check if data is available for the selected pairs.


In [285]:
if 'results_df' in locals() and len(results_df) > 0:
    print("=== EXIT REASON ANALYSIS ===")

    # Exit reasons breakdown
    exit_reasons = results_df['exit_reason'].value_counts()
    exit_reasons_pct = results_df['exit_reason'].value_counts(normalize=True) * 100

    print("Exit reasons breakdown:")
    for reason, count in exit_reasons.items():
        pct = exit_reasons_pct[reason]
        avg_profit = results_df[results_df['exit_reason'] == reason]['profit_ratio'].mean() * 100
        print(f"  {reason}: {count} trades ({pct:.1f}%) - Avg profit: {avg_profit:.2f}%")

    print("\n=== STRATEGY-SPECIFIC ANALYSIS ===")

    # Analyze partial exits (specific to WarriorMomentum strategy)
    partial_exits = results_df[results_df['exit_reason'].str.contains('partial', case=False, na=False)]
    if len(partial_exits) > 0:
        print(f"🔄 Partial exits: {len(partial_exits)} trades")
        print(f"   Average profit from partial exits: {partial_exits['profit_ratio'].mean() * 100:.2f}%")

    # Analyze custom exits (first_red, extension_bar)
    custom_exits = results_df[results_df['exit_reason'].isin(['first_red', 'extension_bar'])]
    if len(custom_exits) > 0:
        print(f"🎯 Custom exits: {len(custom_exits)} trades")
        for reason in ['first_red', 'extension_bar']:
            reason_trades = results_df[results_df['exit_reason'] == reason]
            if len(reason_trades) > 0:
                avg_profit = reason_trades['profit_ratio'].mean() * 100
                print(f"   {reason}: {len(reason_trades)} trades - Avg profit: {avg_profit:.2f}%")

    # Analyze stop losses
    stop_losses = results_df[results_df['exit_reason'].str.contains('stop', case=False, na=False)]
    if len(stop_losses) > 0:
        print(f"🛑 Stop losses: {len(stop_losses)} trades")
        print(f"   Average loss from stop losses: {stop_losses['profit_ratio'].mean() * 100:.2f}%")

    # Analyze ROI exits
    roi_exits = results_df[results_df['exit_reason'] == 'roi']
    if len(roi_exits) > 0:
        print(f"💰 ROI exits: {len(roi_exits)} trades")
        print(f"   Average profit from ROI exits: {roi_exits['profit_ratio'].mean() * 100:.2f}%")

    print("\n=== TRADE TIMING ANALYSIS ===")

    # Convert dates for analysis
    results_df['open_date'] = pd.to_datetime(results_df['open_date'])
    results_df['close_date'] = pd.to_datetime(results_df['close_date'])

    # Hour of day analysis
    results_df['open_hour'] = results_df['open_date'].dt.hour
    hourly_performance = results_df.groupby('open_hour').agg({
        'profit_ratio': ['count', 'mean'],
        'profit_abs': 'sum'
    }).round(4)

    hourly_performance.columns = ['Trades', 'Avg_Profit_%', 'Total_Profit']
    hourly_performance['Avg_Profit_%'] *= 100

    print("Performance by hour of day (top 5):")
    top_hours = hourly_performance.sort_values('Total_Profit', ascending=False).head(5)
    print(top_hours.to_string())

    # Day of week analysis
    results_df['open_day'] = results_df['open_date'].dt.day_name()
    daily_performance = results_df.groupby('open_day').agg({
        'profit_ratio': ['count', 'mean'],
        'profit_abs': 'sum'
    }).round(4)

    daily_performance.columns = ['Trades', 'Avg_Profit_%', 'Total_Profit']
    daily_performance['Avg_Profit_%'] *= 100

    print("\nPerformance by day of week:")
    print(daily_performance.to_string())

else:
    print("⚠️ No results available for exit reason analysis.")


⚠️ No results available for exit reason analysis.


In [286]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

if 'results_df' in locals() and len(results_df) > 0:

    # 1. Cumulative Returns Chart
    fig_returns = go.Figure()
    fig_returns.add_trace(go.Scatter(
        x=results_df['close_date'],
        y=results_df['cumulative_profit'],
        mode='lines',
        name='Cumulative Profit',
        line=dict(color='green', width=2)
    ))
    fig_returns.update_layout(
        title='WarriorMomentum Strategy - Cumulative Profit Over Time',
        xaxis_title='Date',
        yaxis_title=f'Cumulative Profit ({config_backtest["stake_currency"]})',
        hovermode='x unified',
        height=500
    )
    fig_returns.show()

    # 2. Trade Profit Distribution
    fig_dist = px.histogram(
        results_df,
        x='profit_ratio',
        nbins=30,
        title='Trade Profit Distribution',
        labels={'profit_ratio': 'Profit Ratio', 'count': 'Number of Trades'}
    )
    fig_dist.add_vline(x=0, line_dash="dash", line_color="red", annotation_text="Break-even")
    fig_dist.update_layout(height=400)
    fig_dist.show()

    # 3. Performance by Pair (Top 15)
    if len(pair_stats) > 0:
        top_pairs = pair_stats.head(15)
        fig_pairs = px.bar(
            x=top_pairs.index,
            y=top_pairs['Total_Profit_Abs'],
            title='Top 15 Pairs by Total Profit',
            labels={'x': 'Trading Pair', 'y': f'Total Profit ({config_backtest["stake_currency"]})'}
        )
        fig_pairs.update_layout(
            xaxis_tickangle=-45,
            height=500
        )
        fig_pairs.show()

    # 4. Exit Reasons Pie Chart
    fig_exit = px.pie(
        values=exit_reasons.values,
        names=exit_reasons.index,
        title='Exit Reasons Distribution'
    )
    fig_exit.update_layout(height=500)
    fig_exit.show()

    # 5. Monthly Performance (if we have enough data)
    if len(results_df) > 10:
        results_df['month'] = results_df['close_date'].dt.to_period('M')
        monthly_profit = results_df.groupby('month')['profit_abs'].sum().reset_index()
        monthly_profit['month'] = monthly_profit['month'].astype(str)

        fig_monthly = px.bar(
            monthly_profit,
            x='month',
            y='profit_abs',
            title='Monthly Profit',
            labels={'profit_abs': f'Monthly Profit ({config_backtest["stake_currency"]})', 'month': 'Month'}
        )
        fig_monthly.update_layout(height=400)
        fig_monthly.show()

    # 6. Drawdown Chart
    fig_dd = go.Figure()
    fig_dd.add_trace(go.Scatter(
        x=results_df['close_date'],
        y=drawdown,
        mode='lines',
        name='Drawdown',
        fill='tonexty',
        line=dict(color='red', width=1)
    ))
    fig_dd.update_layout(
        title='Drawdown Over Time',
        xaxis_title='Date',
        yaxis_title=f'Drawdown ({config_backtest["stake_currency"]})',
        hovermode='x unified',
        height=400
    )
    fig_dd.show()

    # 7. Win Rate by Pair (Top 15)
    if len(pair_stats) > 0:
        top_pairs_winrate = pair_stats.sort_values('Win_Rate_%', ascending=False).head(15)
        fig_winrate = px.bar(
            x=top_pairs_winrate.index,
            y=top_pairs_winrate['Win_Rate_%'],
            title='Top 15 Pairs by Win Rate',
            labels={'x': 'Trading Pair', 'y': 'Win Rate (%)'}
        )
        fig_winrate.update_layout(
            xaxis_tickangle=-45,
            height=500
        )
        fig_winrate.show()

    # 8. Trade Duration Distribution
    if 'trade_duration' in results_df.columns:
        fig_duration = px.histogram(
            results_df,
            x='trade_duration',
            nbins=20,
            title='Trade Duration Distribution',
            labels={'trade_duration': 'Trade Duration (minutes)', 'count': 'Number of Trades'}
        )
        fig_duration.update_layout(height=400)
        fig_duration.show()

    print("📊 All visualizations have been generated!")
    print("Scroll up to see the charts showing:")
    print("  - Cumulative profit over time")
    print("  - Trade profit distribution")
    print("  - Performance by pair")
    print("  - Exit reasons breakdown")
    print("  - Monthly performance")
    print("  - Drawdown analysis")
    print("  - Win rates by pair")
    print("  - Trade duration distribution")

else:
    print("⚠️ No results available for visualization.")
    print("Please run the backtest first to generate charts.")


⚠️ No results available for visualization.
Please run the backtest first to generate charts.


In [287]:
if 'results_df' in locals() and len(results_df) > 0:
    print("=== STRATEGY OPTIMIZATION SUGGESTIONS ===")

    # 1. Pair Selection Optimization
    print("\n1. 📊 PAIR SELECTION OPTIMIZATION:")
    if len(pair_stats) > 0:
        # Identify consistently profitable pairs
        profitable_pairs = pair_stats[pair_stats['Total_Profit_Abs'] > 0]
        high_winrate_pairs = pair_stats[pair_stats['Win_Rate_%'] > 60]

        print(f"   • {len(profitable_pairs)} out of {len(pair_stats)} pairs are profitable")
        print(f"   • {len(high_winrate_pairs)} pairs have win rate > 60%")

        if len(profitable_pairs) > 0:
            print(f"   • Consider focusing on top {min(10, len(profitable_pairs))} profitable pairs")
            print(f"   • Best performing pair: {profitable_pairs.index[0]} "
                  f"({profitable_pairs.iloc[0]['Total_Profit_Abs']:.2f} {config_backtest['stake_currency']})")

    # 2. Exit Strategy Analysis
    print("\n2. 🚪 EXIT STRATEGY OPTIMIZATION:")
    if len(exit_reasons) > 0:
        roi_trades = len(results_df[results_df['exit_reason'] == 'roi'])
        stop_trades = len(results_df[results_df['exit_reason'].str.contains('stop', case=False, na=False)])

        if roi_trades > 0:
            roi_profit = results_df[results_df['exit_reason'] == 'roi']['profit_ratio'].mean() * 100
            print(f"   • ROI exits: {roi_trades} trades with avg profit {roi_profit:.2f}%")
            if roi_profit < 8:  # Less than expected 10% ROI
                print("   • Consider lowering ROI target for more exits")

        if stop_trades > 0:
            stop_loss = results_df[results_df['exit_reason'].str.contains('stop', case=False, na=False)]['profit_ratio'].mean() * 100
            print(f"   • Stop losses: {stop_trades} trades with avg loss {stop_loss:.2f}%")
            if abs(stop_loss) > 6:  # More than expected 5% stop loss
                print("   • Consider tightening stop loss or improving entry timing")

    # 3. Timing Optimization
    print("\n3. ⏰ TIMING OPTIMIZATION:")
    if 'hourly_performance' in locals():
        best_hours = hourly_performance.sort_values('Total_Profit', ascending=False).head(3)
        worst_hours = hourly_performance.sort_values('Total_Profit', ascending=True).head(3)

        print("   • Best performing hours:")
        for hour in best_hours.index:
            profit = best_hours.loc[hour, 'Total_Profit']
            trades = best_hours.loc[hour, 'Trades']
            print(f"     - {hour:02d}:00: {profit:.2f} {config_backtest['stake_currency']} ({trades} trades)")

        print("   • Consider avoiding these hours:")
        for hour in worst_hours.index:
            profit = worst_hours.loc[hour, 'Total_Profit']
            if profit < 0:
                trades = worst_hours.loc[hour, 'Trades']
                print(f"     - {hour:02d}:00: {profit:.2f} {config_backtest['stake_currency']} ({trades} trades)")

    # 4. Risk Management
    print("\n4. 🛡️ RISK MANAGEMENT:")
    if 'max_drawdown' in locals():
        print(f"   • Maximum drawdown: {max_drawdown:.2f} {config_backtest['stake_currency']}")
        if abs(max_drawdown) > total_profit * 0.3:  # Drawdown > 30% of total profit
            print("   • Consider reducing position sizes or tightening stop losses")

    if 'sharpe_ratio' in locals():
        print(f"   • Sharpe ratio: {sharpe_ratio:.3f}")
        if sharpe_ratio < 1.0:
            print("   • Consider improving risk-adjusted returns")

    # 5. Volume and Volatility Filters
    print("\n5. 📈 ENTRY CONDITION OPTIMIZATION:")
    print("   • Current strategy requires 2x volume increase")
    print("   • Consider testing different volume multipliers (1.5x, 2.5x, 3x)")
    print("   • Test different volatility thresholds for breakout confirmation")
    print("   • Consider adding momentum indicators (RSI, MACD) for entry confirmation")

    # 6. Position Sizing
    print("\n6. 💰 POSITION SIZING:")
    if config_backtest['stake_amount'] == 'unlimited':
        print("   • Currently using unlimited stake amount")
        print("   • Consider fixed position sizing for better risk management")
    else:
        print(f"   • Current stake amount: {config_backtest['stake_amount']} {config_backtest['stake_currency']}")

    print(f"   • Max open trades: {config_backtest['max_open_trades']}")
    if len(results_df) > 0:
        avg_concurrent = len(results_df) / len(results_df['open_date'].dt.date.unique()) if len(results_df) > 0 else 0
        print(f"   • Average concurrent trades: {avg_concurrent:.1f}")

    print("\n=== NEXT STEPS ===")
    print("1. 🔧 Run hyperopt to optimize parameters")
    print("2. 📊 Test on different timeframes (1m, 15m)")
    print("3. 🎯 Focus on most profitable pairs")
    print("4. ⚡ Add more momentum indicators")
    print("5. 🛡️ Implement dynamic position sizing")
    print("6. 📈 Test different volume thresholds")

else:
    print("⚠️ No results available for optimization analysis.")
    print("Please run the backtest first to get optimization suggestions.")


⚠️ No results available for optimization analysis.
Please run the backtest first to get optimization suggestions.


In [288]:
print("=== DATA DOWNLOAD COMMANDS ===")
print("If you need to download data for backtesting, use these commands in terminal:")
print()
print("# Download data for top USDT pairs (5m and 1d timeframes)")
print("freqtrade download-data --exchange binance --timeframes 5m 1d --pairs \\")

# Common USDT pairs for download
common_pairs = [
    "BTC/USDT", "ETH/USDT", "BNB/USDT", "ADA/USDT", "XRP/USDT",
    "SOL/USDT", "DOT/USDT", "DOGE/USDT", "AVAX/USDT", "SHIB/USDT",
    "MATIC/USDT", "LTC/USDT", "UNI/USDT", "LINK/USDT", "ATOM/USDT",
    "XLM/USDT", "BCH/USDT", "NEAR/USDT", "ALGO/USDT", "VET/USDT",
    "TRX/USDT", "FTM/USDT", "SAND/USDT", "MANA/USDT", "CRV/USDT"
]

# Print pairs in groups of 5 for readability
for i in range(0, len(common_pairs), 5):
    group = common_pairs[i:i+5]
    print(" ".join(group), end="")
    if i + 5 < len(common_pairs):
        print(" \\")
    else:
        print()

print()
print("# Alternative: Download all available USDT pairs")
print("freqtrade download-data --exchange binance --timeframes 5m 1d --pairs-file user_data/usdt_pairs.txt")
print()
print("# For more exchanges, replace 'binance' with: kraken, okx, bybit, etc.")
print()
print("# Download specific date range (example)")
print("freqtrade download-data --exchange binance --timeframes 5m 1d \\")
print("  --timerange 20240101-20241201 --pairs BTC/USDT ETH/USDT")
print()
print("📝 Note: The WarriorMomentum strategy requires both 5m and 1d data")
print("📊 Recommended: At least 3-6 months of data for meaningful backtesting")
print("⚡ Tip: Start with fewer pairs to test the strategy, then expand")


=== DATA DOWNLOAD COMMANDS ===
If you need to download data for backtesting, use these commands in terminal:

# Download data for top USDT pairs (5m and 1d timeframes)
freqtrade download-data --exchange binance --timeframes 5m 1d --pairs \
BTC/USDT ETH/USDT BNB/USDT ADA/USDT XRP/USDT \
SOL/USDT DOT/USDT DOGE/USDT AVAX/USDT SHIB/USDT \
MATIC/USDT LTC/USDT UNI/USDT LINK/USDT ATOM/USDT \
XLM/USDT BCH/USDT NEAR/USDT ALGO/USDT VET/USDT \
TRX/USDT FTM/USDT SAND/USDT MANA/USDT CRV/USDT

# Alternative: Download all available USDT pairs
freqtrade download-data --exchange binance --timeframes 5m 1d --pairs-file user_data/usdt_pairs.txt

# For more exchanges, replace 'binance' with: kraken, okx, bybit, etc.

# Download specific date range (example)
freqtrade download-data --exchange binance --timeframes 5m 1d \
  --timerange 20240101-20241201 --pairs BTC/USDT ETH/USDT

📝 Note: The WarriorMomentum strategy requires both 5m and 1d data
📊 Recommended: At least 3-6 months of data for meaningful backt